# INITIAL IMPORT

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline


In [2]:
import os
from src.config import Configuration


CONFIG = Configuration(

)

In [3]:
input_path = CONFIG.mr_nf_structural
output_path = CONFIG.mr_nf_tensors
output_96_path = CONFIG.mr_nf_tensors_96
modalities = ['T1', 'T1GD', 'T2', 'FLAIR']
filter_visits = True

# input_path = CONFIG.brats_path_structural
# output_path = CONFIG.brats_tensors
# output_96_path = CONFIG.brats_tensors_96
# modalities = ['t2', 't1ce', 't1', 'flair']
# filter_visits = False


from maikol_utils.file_utils import make_dirs

make_dirs([output_path, output_96_path])

[True, True]

In [4]:
import os
import pickle

import nibabel as nib
import torch
from tqdm import tqdm

from maikol_utils.print_utils import print_warn
from src.config import Configuration

def convert_niigz_to_tensor(CONFIG: Configuration, filter_visits: bool = True):
    """
    Convert patient NIfTI files to tensors.

    Args:
        CONFIG: Project configuration object.
        filter_visits:
            - True: keep one folder per base patient ID, preferring *_11 over *_21.
            - False: parse all patient folders found in input_path.
    """
    selected_entries = []

    if filter_visits:
        # Collect candidate folders by base ID and prefer _11 over _21.
        selected_patients = {}

        for patient_id in os.listdir(input_path):
            patient_path = os.path.join(input_path, patient_id)

            if not os.path.isdir(patient_path):
                print_warn(f"Skipping {patient_path} as it is not a directory")
                continue

            if not (patient_id.endswith('_11') or patient_id.endswith('_21')):
                continue

            base_id, visit_suffix = patient_id.rsplit('_', 1)
            existing = selected_patients.get(base_id)

            if existing is None:
                selected_patients[base_id] = (patient_id, visit_suffix)
                continue

            # Keep _11 whenever present; only use _21 if _11 does not exist.
            _, existing_suffix = existing
            if existing_suffix == '21' and visit_suffix == '11':
                selected_patients[base_id] = (patient_id, visit_suffix)

        selected_entries = [
            (base_id, patient_id)
            for base_id, (patient_id, _) in selected_patients.items()
        ]
    else:
        # Parse every patient directory as-is.
        for patient_id in os.listdir(input_path):
            patient_path = os.path.join(input_path, patient_id)

            if not os.path.isdir(patient_path):
                print_warn(f"Skipping {patient_path} as it is not a directory")
                continue

            selected_entries.append((patient_id, patient_id))

    for output_id, patient_id in tqdm(selected_entries):
        patient_path = os.path.join(input_path, patient_id)

        modality_tensors = {}

        for mod in modalities:
            file_path = os.path.join(patient_path, f"{patient_id}_{mod}.nii.gz")
            img_data = nib.load(file_path).get_fdata()
            modality_tensors[mod] = torch.tensor(img_data, dtype=torch.float32)

        output_file = os.path.join(output_path, f"{output_id}.pkl")
        with open(output_file, "wb") as handle:
            pickle.dump(modality_tensors, handle)


convert_niigz_to_tensor(CONFIG, filter_visits=filter_visits)

100%|██████████| 630/630 [04:14<00:00,  2.47it/s]


# Rescale

In [5]:
import os
import pickle

import torch
import torch.nn.functional as F
from tqdm import tqdm

TARGET = 96

for fname in tqdm(os.listdir(output_path)):
    if not fname.endswith(".pkl"):
        continue

    with open(os.path.join(output_path, fname), "rb") as handle:
        volume_dict = pickle.load(handle)

    resized_volume_dict = {}
    for modality, volume in volume_dict.items():
        resized_volume_dict[modality] = F.interpolate(
            volume.unsqueeze(0).unsqueeze(0),
            size=(TARGET, TARGET, TARGET),
            mode="trilinear",
            align_corners=False,
        ).squeeze(0).squeeze(0)

    with open(os.path.join(output_96_path, fname), "wb") as handle:
        pickle.dump(resized_volume_dict, handle)

100%|██████████| 630/630 [02:10<00:00,  4.84it/s]
